# 05 · Statistical analysis

Reproduces Tables 7–9: paired $t$-tests comparing adjacent commitment levels (low vs. medium, medium vs. high) for each model and clause type. The test is on the rate of giving the answer favoured by the supportive nudge (PP$^+$), with the 500 base prompts as paired observations; pairs where either answer could not be parsed are dropped. $t$ is computed as higher minus lower commitment, so positive values mean more agreement at the higher level.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path("..").resolve()
sys.path.insert(0, str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = DATA_DIR / "outputs"
FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
EXPECTED_DIR = BASE_DIR / "expected"

FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

In [2]:
import pandas as pd

from sway import stats
from sway.checks import compare

In [3]:
MODEL_FILE_KEYS = {
    "meta": "llama4_scout",
    "mistral": "mistral_large",
    "claude_sonnet": "claude_sonnet",
    "claude_opus": "claude_opus",
    "claude_haiku": "claude_haiku",
    "gemma-3-4b-it": "gemma_3_4b",
}

AITA_FILES = {
    "meta": OUTPUT_DIR / "aita" / "meta_results.csv",
    "mistral": OUTPUT_DIR / "aita" / "mistral_results.csv",
    "claude_sonnet": OUTPUT_DIR / "aita" / "claude_sonnet_results.csv",
    "claude_opus": OUTPUT_DIR / "aita" / "claude_opus_results.csv",
    "claude_haiku": OUTPUT_DIR / "aita" / "claude_haiku_results.csv",
    "gemma-3-4b-it": OUTPUT_DIR / "aita" / "gemma-3-4b-it_results.csv",
}


def model_files(folder):
    return {m: OUTPUT_DIR / folder / f"{key}_results.csv" for m, key in MODEL_FILE_KEYS.items()}


LFQA_FILES = model_files("lfqa")
DQA_FILES = model_files("dqa")

In [4]:
ttests = pd.concat([
    stats.paired_ttests_pplus(AITA_FILES, "AITA"),
    stats.paired_ttests_pplus(LFQA_FILES, "LFQA"),
    stats.paired_ttests_pplus(DQA_FILES, "DQA"),
], ignore_index=True)
ttests.to_csv(TABLE_DIR / "tables_7_9_paired_ttests.csv", index=False)

compare(ttests, pd.read_csv(EXPECTED_DIR / "ttest_pplus_recomputed.csv"), ["dataset", "model", "group", "trans"], ["t_high_minus_low", "p", "n_pairs", "mean_diff"], label="Tables 7-9")
ttests.groupby("dataset").n_pairs.agg(["min", "max"])

[OK] Tables 7-9: 144 rows matched, 0 unmatched, max abs diff 1.78e-15 (tolerance 1e-09)


,min,max
dataset,,
AITA,493,500
DQA,493,500
LFQA,490,500


## LaTeX rows for Tables 7–9

In [5]:
MODEL_ORDER = ["meta", "mistral", "claude_sonnet", "claude_opus", "claude_haiku", "gemma-3-4b-it"]
MODEL_SHORT = {"meta": "Llama", "mistral": "Mistral", "claude_sonnet": "C. Sonnet", "claude_opus": "C. Opus", "claude_haiku": "C. Haiku", "gemma-3-4b-it": "Gemma"}

for ds, tex in [("AITA", "aita"), ("LFQA", "lfqa"), ("DQA", "dqa")]:
    rows = stats.ttest_latex_rows(ttests[ttests.dataset == ds], MODEL_ORDER, MODEL_SHORT)
    (TABLE_DIR / f"ttest_rows_{tex}.tex").write_text(rows + "\n")
    expected = (EXPECTED_DIR / f"ttest_rows_{tex}.tex").read_text().strip()
    print(f"[{'OK' if rows.strip() == expected else 'MISMATCH'}] {ds} LaTeX rows identical to the paper")
    assert rows.strip() == expected
    print(rows, end="\n\n")

[OK] AITA LaTeX rows identical to the paper
Llama & $7.25^{***}$ & $-1.70$ & $-0.13$ & $-3.19^{**}$ & $6.44^{***}$ & $5.53^{***}$ & $-1.18$ & $-7.16^{***}$ \\
Mistral & $5.18^{***}$ & $2.81^{**}$ & $3.59^{***}$ & $2.54^{*}$ & $6.77^{***}$ & $-0.34$ & $4.91^{***}$ & $-3.02^{**}$ \\
C. Sonnet & $3.30^{**}$ & $-0.36$ & $0.26$ & $-1.53$ & $7.61^{***}$ & $11.30^{***}$ & $0.83$ & $-3.33^{***}$ \\
C. Opus & $3.33^{***}$ & $-0.65$ & $3.57^{***}$ & $-2.13^{*}$ & $4.58^{***}$ & $9.73^{***}$ & $3.19^{**}$ & $-6.04^{***}$ \\
C. Haiku & $2.46^{*}$ & $-0.93$ & $4.20^{***}$ & $-4.11^{***}$ & $1.46$ & $4.73^{***}$ & $-1.74$ & $-1.71$ \\
Gemma & $4.07^{***}$ & $10.26^{***}$ & $-0.11$ & $4.80^{***}$ & $5.04^{***}$ & $6.01^{***}$ & $0.63$ & $6.34^{***}$ \\

[OK] LFQA LaTeX rows identical to the paper
Llama & $9.46^{***}$ & $-9.38^{***}$ & $-2.07^{*}$ & $-5.63^{***}$ & $7.30^{***}$ & $-7.12^{***}$ & $-0.41$ & $-6.53^{***}$ \\
Mistral & $9.90^{***}$ & $-5.44^{***}$ & $2.32^{*}$ & $-1.89$ & $1.28$ & $10.04^